# SBERT STS-B Fine-Tuning — 10 Seed Experiment

This notebook runs the **SBERT-NLI-STSb-base** fine-tuning experiment across **10 random seeds**.

### Purpose

The Sentence-BERT paper reports:

- **SBERT-NLI-STSb-base:** **85.35 ± 0.17** Spearman on STS-B.
- The paper reports results averaged over **10 random seeds**.
- STS-B fine-tuning uses the regression objective: cosine similarity between the two sentence embeddings, optimized with MSE.

This notebook repeats the same experiment structure for 10 seeds and reports:

- test Spearman for every seed
- mean
- standard deviation
- minimum / maximum
- difference from the paper's reported mean of 85.35

### Important implementation detail

The NLI checkpoint used here is **our locally trained SBERT-NLI-base checkpoint**, not the original authors' checkpoint. Therefore, this is a **10-seed reproduction using our NLI model**, not an exact reproduction of the authors' original training run.

Hardware adaptation retained from the single-seed notebook:

- physical batch size = 4
- gradient accumulation = 4
- effective batch size = 16
- max sequence length = 128
- AMP enabled on CUDA

The paper explicitly reports 4 epochs for the STS-B fine-tuning experiment, while some other STS-B hyperparameters are not specified in the paper text; those implementation choices are kept consistent with the previous notebook.

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import math
import random
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModel
from scipy.stats import spearmanr
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

f:\Programs\PYTHON_310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
f:\Programs\PYTHON_310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-uncased"

DATA_DIR = Path("..") / "data"
STS_DIR = DATA_DIR / "STS"

NLI_MODEL_DIR = Path("outputs") / "sbert_nli_base"

NLI_MODEL_PATH = (
    NLI_MODEL_DIR
    / "model"
    / "sbert_nli_base.pt"
)

NLI_TOKENIZER_PATH = (
    NLI_MODEL_DIR
    / "tokenizer"
)

# Separate output directory so this experiment does not
# overwrite the previous single-seed experiment.
OUTPUT_DIR = Path("outputs") / "sbert_nli_stsb_10_seeds"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED_CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SEED_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / "10_seed_results.csv"

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 4
LEARNING_RATE = 2e-5

MAX_LENGTH = 128

PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

EFFECTIVE_BATCH_SIZE = (
    PHYSICAL_BATCH_SIZE *
    GRADIENT_ACCUMULATION_STEPS
)

EVAL_BATCH_SIZE = 32

# The paper reports 10 random seeds.
SEEDS = list(range(42, 52))

PAPER_SCORE = 85.35

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = torch.cuda.is_available()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)

print("\nEpochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

print("\nPhysical batch size:", PHYSICAL_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)

print("\nMax length:", MAX_LENGTH)
print("Seeds:", SEEDS)
print("Number of seeds:", len(SEEDS))
print("Paper score:", PAPER_SCORE)

Model: bert-base-uncased
Device: cuda

Epochs: 4
Learning rate: 2e-05

Physical batch size: 4
Gradient accumulation: 4
Effective batch size: 16

Max length: 128
Seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
Number of seeds: 10
Paper score: 85.35


In [3]:
# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Do not force fully deterministic CUDA kernels because
    # that can reduce performance and is not required by the
    # paper's "10 random seeds" protocol.
    torch.backends.cudnn.benchmark = False


print("Seed function ready.")

Seed function ready.


In [4]:
# ============================================================
# 4. LOAD STS-B
# ============================================================

STSB_PATH = STS_DIR / "STSb"

if not STSB_PATH.exists():
    raise FileNotFoundError(
        f"STS-B dataset not found:\n{STSB_PATH.resolve()}"
    )

stsb_dataset = load_from_disk(STSB_PATH)

print(stsb_dataset)

for split in stsb_dataset.keys():
    print(
        f"{split:12s}: "
        f"{len(stsb_dataset[split]):,} examples"
    )

DatasetDict({
    train: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2'],
        num_rows: 1379
    })
})
train       : 5,749 examples
validation  : 1,500 examples
test        : 1,379 examples


In [5]:
# ============================================================
# 5. CONVERT STS-B SPLITS
# ============================================================

train_df = stsb_dataset["train"].to_pandas()
dev_df = stsb_dataset["validation"].to_pandas()
test_df = stsb_dataset["test"].to_pandas()

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())

Train: (5749, 8)
Dev: (1500, 8)
Test: (1379, 8)

Columns: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']


In [6]:
# ============================================================
# 6. LOAD TOKENIZER + NLI CHECKPOINT
# ============================================================

if not NLI_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"NLI checkpoint not found:\n{NLI_MODEL_PATH.resolve()}"
    )

if not NLI_TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"NLI tokenizer not found:\n{NLI_TOKENIZER_PATH.resolve()}"
    )

tokenizer = AutoTokenizer.from_pretrained(
    NLI_TOKENIZER_PATH
)

# The checkpoint was created by us, so weights_only=False is
# appropriate here. The checkpoint contains metadata in addition
# to the model state dict.
nli_checkpoint = torch.load(
    NLI_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

model_state_dict = nli_checkpoint["model_state_dict"]

bert_state_dict = {
    key[len("bert."):]: value
    for key, value in model_state_dict.items()
    if key.startswith("bert.")
}

print("NLI checkpoint loaded.")
print("BERT parameters:", len(bert_state_dict))

NLI checkpoint loaded.
BERT parameters: 199


In [7]:
# ============================================================
# 7. STS-B DATASET
# ============================================================

class STSBDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe):
        self.sentence1 = dataframe["sentence1"].astype(str).tolist()
        self.sentence2 = dataframe["sentence2"].astype(str).tolist()
        self.scores = dataframe["score"].astype(float).tolist()

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return {
            "sentence1": self.sentence1[idx],
            "sentence2": self.sentence2[idx],
            "score": self.scores[idx]
        }


train_dataset = STSBDataset(train_df)
dev_dataset = STSBDataset(dev_df)
test_dataset = STSBDataset(test_df)

print("Train:", len(train_dataset))
print("Dev:", len(dev_dataset))
print("Test:", len(test_dataset))

Train: 5749
Dev: 1500
Test: 1379


In [8]:
# ============================================================
# 8. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    sentences1 = [
        item["sentence1"]
        for item in batch
    ]

    sentences2 = [
        item["sentence2"]
        for item in batch
    ]

    scores = torch.tensor(
        [
            item["score"]
            for item in batch
        ],
        dtype=torch.float
    )

    inputs1 = tokenizer(
        sentences1,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs2 = tokenizer(
        sentences2,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        "sentence1": inputs1,
        "sentence2": inputs2,
        "scores": scores
    }


print("Collate function ready.")

Collate function ready.


In [9]:
# ============================================================
# 9. MODEL DEFINITION
# ============================================================

class SBERTSTSRegressor(nn.Module):

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def mean_pooling(
        self,
        last_hidden_state,
        attention_mask
    ):
        mask = attention_mask.unsqueeze(-1).expand(
            last_hidden_state.size()
        ).float()

        summed = torch.sum(
            last_hidden_state * mask,
            dim=1
        )

        counts = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        return summed / counts

    def encode(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        return self.mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

    def forward(
        self,
        sentence1_inputs,
        sentence2_inputs
    ):
        u = self.encode(
            sentence1_inputs["input_ids"],
            sentence1_inputs["attention_mask"]
        )

        v = self.encode(
            sentence2_inputs["input_ids"],
            sentence2_inputs["attention_mask"]
        )

        return torch.nn.functional.cosine_similarity(
            u,
            v,
            dim=1
        )


print("Model definition ready.")

Model definition ready.


In [10]:
# ============================================================
# 10. SCORE NORMALIZATION
# ============================================================

def normalize_stsb_scores(scores):
    """
    STS-B labels are in [0, 5].
    Convert them to [0, 1] for the cosine-regression target.
    """
    return scores / 5.0


print("STS-B score normalization: [0, 5] -> [0, 1]")

STS-B score normalization: [0, 5] -> [0, 1]


In [11]:
# ============================================================
# 11. MODEL FACTORY
# ============================================================

def build_model():
    """
    Build a fresh BERT encoder for each seed and load exactly
    the same NLI-trained SBERT weights.

    Every seed starts from the same NLI checkpoint. The seed
    changes stochastic fine-tuning behavior.
    """

    encoder = AutoModel.from_pretrained(
        MODEL_NAME
    )

    missing_keys, unexpected_keys = encoder.load_state_dict(
        bert_state_dict,
        strict=False
    )

    if missing_keys or unexpected_keys:
        raise RuntimeError(
            "NLI BERT checkpoint did not load cleanly.\n"
            f"Missing: {missing_keys}\n"
            f"Unexpected: {unexpected_keys}"
        )

    model = SBERTSTSRegressor(
        encoder
    ).to(DEVICE)

    return model


print("Model factory ready.")

Model factory ready.


In [12]:
# ============================================================
# 12. DATALOADERS
# ============================================================

def build_dataloaders(seed):

    # Explicit generator makes the training shuffle seed
    # reproducible for each experiment.
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=PHYSICAL_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, dev_loader, test_loader


print("Dataloader factory ready.")

Dataloader factory ready.


In [13]:
# ============================================================
# 13. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    scaler,
    device,
    accumulation_steps
):

    model.train()

    total_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(
        dataloader,
        desc="Training",
        leave=False
    )

    for step, batch in enumerate(progress_bar):

        inputs1 = {
            key: value.to(device)
            for key, value in batch["sentence1"].items()
        }

        inputs2 = {
            key: value.to(device)
            for key, value in batch["sentence2"].items()
        }

        scores = normalize_stsb_scores(
            batch["scores"]
        ).to(device)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):
            predictions = model(
                inputs1,
                inputs2
            )

            loss = criterion(
                predictions,
                scores
            )

            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        should_update = (
            (step + 1) % accumulation_steps == 0
            or (step + 1) == len(dataloader)
        )

        if should_update:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accumulation_steps

        progress_bar.set_postfix(
            loss=f"{loss.item() * accumulation_steps:.4f}"
        )

    return total_loss / len(dataloader)

In [14]:
# ============================================================
# 14. VALIDATION / TEST EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_stsb(
    model,
    dataloader,
    device
):

    model.eval()

    predictions = []
    gold_scores = []

    for batch in tqdm(
        dataloader,
        desc="Evaluating",
        leave=False
    ):

        inputs1 = {
            key: value.to(device)
            for key, value in batch["sentence1"].items()
        }

        inputs2 = {
            key: value.to(device)
            for key, value in batch["sentence2"].items()
        }

        scores = batch["scores"].numpy()

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):
            similarity = model(
                inputs1,
                inputs2
            )

        similarity = (
            similarity
            .float()
            .cpu()
            .numpy()
        )

        predictions.extend(
            similarity.tolist()
        )

        # Dividing by 5 does not change Spearman because it
        # preserves the ranking of the gold labels.
        gold_scores.extend(
            (scores / 5.0).tolist()
        )

    predictions = np.array(predictions)
    gold_scores = np.array(gold_scores)

    rho = spearmanr(
        predictions,
        gold_scores
    ).statistic

    return rho * 100

In [15]:
# ============================================================
# 15. SINGLE-SEED EXPERIMENT
# ============================================================

def run_seed(seed):

    print("\n" + "=" * 80)
    print(f"STARTING SEED {seed}")
    print("=" * 80)

    set_seed(seed)

    train_loader, dev_loader, test_loader = build_dataloaders(seed)

    model = build_model()

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_dev_score = -float("inf")
    best_epoch = None
    history = []

    seed_checkpoint_path = (
        SEED_CHECKPOINT_DIR /
        f"sbert_nli_stsb_seed_{seed}.pt"
    )

    start_time = time.time()

    for epoch in range(EPOCHS):

        print("\n" + "-" * 70)
        print(f"Seed {seed} | Epoch {epoch + 1}/{EPOCHS}")
        print("-" * 70)

        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=DEVICE,
            accumulation_steps=GRADIENT_ACCUMULATION_STEPS
        )

        dev_spearman = evaluate_stsb(
            model=model,
            dataloader=dev_loader,
            device=DEVICE
        )

        history.append({
            "seed": seed,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "dev_spearman": dev_spearman
        })

        print(f"Train loss: {train_loss:.6f}")
        print(f"Dev Spearman: {dev_spearman:.4f}")

        if dev_spearman > best_dev_score:

            best_dev_score = dev_spearman
            best_epoch = epoch + 1

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": MODEL_NAME,
                    "pooling": "mean",
                    "max_length": MAX_LENGTH,
                    "learning_rate": LEARNING_RATE,
                    "epochs": EPOCHS,
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "best_dev_spearman": best_dev_score,
                },
                seed_checkpoint_path
            )

            print(
                f"Best checkpoint saved "
                f"(epoch {best_epoch})"
            )

    # --------------------------------------------------------
    # Load best checkpoint for this seed
    # --------------------------------------------------------

    best_checkpoint = torch.load(
        seed_checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    model.eval()

    test_spearman = evaluate_stsb(
        model=model,
        dataloader=test_loader,
        device=DEVICE
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60.0

    result = {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_dev_spearman": best_dev_score,
        "test_spearman": test_spearman,
        "difference_from_paper": test_spearman - PAPER_SCORE,
        "runtime_minutes": elapsed_minutes
    }

    history_df = pd.DataFrame(history)

    history_path = (
        OUTPUT_DIR /
        f"history_seed_{seed}.csv"
    )

    history_df.to_csv(
        history_path,
        index=False
    )

    # --------------------------------------------------------
    # Free GPU memory before the next seed
    # --------------------------------------------------------

    del model
    del optimizer
    del scaler
    del train_loader
    del dev_loader
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\nSeed result:")
    print(
        f"Seed {seed}: "
        f"test Spearman = {test_spearman:.4f}"
    )

    print(
        f"Runtime: {elapsed_minutes:.2f} minutes"
    )

    return result

In [16]:
# ============================================================
# 16. RUN ALL 10 SEEDS
# ============================================================

all_results = []

for seed in SEEDS:

    result = run_seed(seed)

    all_results.append(result)

    # Save after EVERY seed so that a long experiment is not
    # lost if the notebook stops before all 10 runs finish.
    results_df = pd.DataFrame(all_results)

    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    print("\nCurrent results:")
    display(results_df)

print("\n10-seed experiment complete.")


STARTING SEED 42


W0924 05:58:41.315000 10456 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



----------------------------------------------------------------------
Seed 42 | Epoch 1/4
----------------------------------------------------------------------


Train loss: 0.031887
Dev Spearman: 85.9164
Best checkpoint saved (epoch 1)

----------------------------------------------------------------------
Seed 42 | Epoch 2/4
----------------------------------------------------------------------


Train loss: 0.012213
Dev Spearman: 86.0651
Best checkpoint saved (epoch 2)

----------------------------------------------------------------------
Seed 42 | Epoch 3/4
----------------------------------------------------------------------


...

Train loss: 0.007082
Dev Spearman: 86.6521
Best checkpoint saved (epoch 3)

----------------------------------------------------------------------
Seed 51 | Epoch 4/4
----------------------------------------------------------------------


Train loss: 0.005185
Dev Spearman: 86.8294
Best checkpoint saved (epoch 4)



Seed result:
Seed 51: test Spearman = 84.8032
Runtime: 9.75 minutes

Current results:


,seed,best_epoch,best_dev_spearman,test_spearman,difference_from_paper,runtime_minutes
0,42,4,86.604225,84.471953,-0.878047,9.187636
1,43,4,86.444431,84.323734,-1.026266,9.037125
2,44,4,86.415619,84.778752,-0.571248,8.929841
3,45,3,86.809433,84.267421,-1.082579,9.813706
4,46,4,86.715362,84.549910,-0.800090,8.867705
5,47,4,86.945434,84.217830,-1.132170,8.823011
6,48,3,86.493893,84.284987,-1.065013,8.885953
7,49,3,86.546245,84.552604,-0.797396,8.797578
8,50,3,86.800621,84.845598,-0.504402,11.489002
9,51,4,86.829363,84.803242,-0.546758,9.753917



10-seed experiment complete.


In [17]:
# ============================================================
# 17. 10-SEED RESULTS
# ============================================================

results_df = pd.read_csv(
    RESULTS_PATH
)

results_df = results_df.sort_values(
    "seed"
).reset_index(drop=True)

results_df

,seed,best_epoch,best_dev_spearman,test_spearman,difference_from_paper,runtime_minutes
0,42,4,86.604225,84.471953,-0.878047,9.187636
1,43,4,86.444431,84.323734,-1.026266,9.037125
2,44,4,86.415619,84.778752,-0.571248,8.929841
3,45,3,86.809433,84.267421,-1.082579,9.813706
4,46,4,86.715362,84.549910,-0.800090,8.867705
5,47,4,86.945434,84.217830,-1.132170,8.823011
6,48,3,86.493893,84.284987,-1.065013,8.885953
7,49,3,86.546245,84.552604,-0.797396,8.797578
8,50,3,86.800621,84.845598,-0.504402,11.489002
9,51,4,86.829363,84.803242,-0.546758,9.753917


In [18]:
# ============================================================
# 18. SUMMARY STATISTICS
# ============================================================

test_scores = results_df["test_spearman"]

summary = {
    "n_seeds": len(test_scores),
    "mean": test_scores.mean(),
    "std": test_scores.std(ddof=1),
    "min": test_scores.min(),
    "max": test_scores.max(),
    "paper_mean": PAPER_SCORE,
    "difference_mean_vs_paper": test_scores.mean() - PAPER_SCORE
}

summary_df = pd.DataFrame(
    [summary]
)

summary_df

,n_seeds,mean,std,min,max,paper_mean,difference_mean_vs_paper
0,10,84.509603,0.236932,84.21783,84.845598,85.35,-0.840397


In [19]:
# ============================================================
# 19. PAPER TABLE 2 COMPARISON
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "SBERT-NLI-STSb-base",
        "Paper Mean": PAPER_SCORE,
        "Our 10-Seed Mean": test_scores.mean(),
        "Our 10-Seed Std": test_scores.std(ddof=1),
        "Difference": test_scores.mean() - PAPER_SCORE
    }
])

comparison

,Model,Paper Mean,Our 10-Seed Mean,Our 10-Seed Std,Difference
0,SBERT-NLI-STSb-base,85.35,84.509603,0.236932,-0.840397


## Interpretation

The key number to compare with the paper is the **10-seed mean**, not any individual seed.

The paper reports **85.35 ± 0.17**. This experiment should therefore be interpreted in two stages:

1. **Per-seed results:** show how much the result varies with the random seed.
2. **10-seed mean ± standard deviation:** this is the directly comparable statistic to the paper's Table 2 protocol.

A deviation from 85.35 does not by itself identify the cause. Potential sources include:

- our independently trained NLI checkpoint rather than the authors' original checkpoint
- STS-B data/preprocessing differences
- unspecified STS-B implementation details
- maximum sequence length
- batch/gradient-accumulation implementation
- optimizer/training details
- random-seed variation